In [7]:
import pandas as pd

keu  = pd.read_csv("HASIL_KNN_EUCLIDEAN.csv")
kman = pd.read_csv("HASIL_KNN_MANHATTAN.csv")

test = pd.read_csv("DATASET1000_METPEN.csv")


In [3]:
label_map = {"negatif": 1, "positif": 0}

# Ganti 'emosi' kalau nama kolom label Anda berbeda
if "label" in test.columns:
    y_true = test["label"]
elif "emosi" in test.columns:
    # kalau emosi teks, map
    if test["emosi"].dtype == object:
        y_true = test["emosi"].str.lower().map(label_map)
    else:
        y_true = test["emosi"]
else:
    raise ValueError("DATAUJI100_METPEN.csv tidak memiliki kolom label (emosi/label). Evaluasi tidak bisa dilakukan.")

print("=== CEK LABEL ASLI (y_true) ===")
print("Jumlah label:", y_true.shape[0])
print("Unique:", sorted(pd.Series(y_true).dropna().unique().tolist()))
print("5 label pertama:", y_true.head().tolist())


=== CEK LABEL ASLI (y_true) ===
Jumlah label: 1000
Unique: [0, 1]
5 label pertama: [1, 1, 1, 1, 1]


In [4]:
# pastikan file prediksi punya kolom 'id'
key = "id"

df = test[[key]].copy()
df["y_true"] = y_true.values

df = df.merge(keu[[key, "prediksi_emosi"]].rename(columns={"prediksi_emosi":"pred_knn_euclid"}), on=key, how="inner")
df = df.merge(kman[[key, "prediksi_emosi"]].rename(columns={"prediksi_emosi":"pred_knn_manhattan"}), on=key, how="inner")

print("=== CEK HASIL MERGE ===")
print("Shape:", df.shape)
print(df)


=== CEK HASIL MERGE ===
Shape: (100, 4)
     id  y_true  pred_knn_euclid  pred_knn_manhattan
0    11       1                1                   1
1    24       1                1                   1
2    31       1                1                   1
3    40       1                1                   1
4    55       1                1                   1
..  ...     ...              ...                 ...
95  948       0                0                   1
96  974       1                1                   1
97  975       0                1                   1
98  987       1                1                   1
99  999       1                1                   1

[100 rows x 4 columns]


In [5]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def evaluate(y_true, y_pred, name):
    cm = confusion_matrix(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n===== {name} =====")
    print("Confusion Matrix [[TN FP],[FN TP]]:")
    print(cm)
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")

    return {"Model": name, "Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1}

results = []
results.append(evaluate(df["y_true"], df["pred_knn_euclid"], "KNN Euclidean (k=7)"))
results.append(evaluate(df["y_true"], df["pred_knn_manhattan"], "KNN Manhattan/CityBlock (k=7)"))



===== KNN Euclidean (k=7) =====
Confusion Matrix [[TN FP],[FN TP]]:
[[ 9  3]
 [ 1 87]]
Accuracy : 0.9600
Precision: 0.9667
Recall   : 0.9886
F1-score : 0.9775

===== KNN Manhattan/CityBlock (k=7) =====
Confusion Matrix [[TN FP],[FN TP]]:
[[ 4  8]
 [ 5 83]]
Accuracy : 0.8700
Precision: 0.9121
Recall   : 0.9432
F1-score : 0.9274


In [6]:
eval_table = pd.DataFrame(results).sort_values(by="F1-score", ascending=False)
print("\n=== TABEL PERBANDINGAN (sorted by F1) ===")
display(eval_table)



=== TABEL PERBANDINGAN (sorted by F1) ===


,Model,Accuracy,Precision,Recall,F1-score
0,KNN Euclidean (k=7),0.96,0.966667,0.988636,0.977528
1,KNN Manhattan/CityBlock (k=7),0.87,0.912088,0.943182,0.927374
